In [1]:
import math
from transformers import pipeline

# Dataset
satisfied = [
    "service was quick and helpful",
    "happy with customer support",
    "support team was polite",
    "good service experience",
    "problem solved quickly"
]

unsatisfied = [
    "very slow response",
    "issue not resolved",
    "poor customer care",
    "worst support ever",
    "not helpful at all"
]

# Word count
def word_count(data):
    d = {}
    for s in data:
        for w in s.split():
            d[w] = d.get(w, 0) + 1
    return d

sat_words = word_count(satisfied)
unsat_words = word_count(unsatisfied)

# Priors
total = len(satisfied) + len(unsatisfied)
p_sat = len(satisfied) / total
p_unsat = len(unsatisfied) / total

# Vocabulary
vocab = set(sat_words) | set(unsat_words)
V = len(vocab)

sat_total = sum(sat_words.values())
unsat_total = sum(unsat_words.values())  # ✅ fixed

# Naive Bayes
def predict_nb(text):
    s_score = math.log(p_sat)
    u_score = math.log(p_unsat)

    for w in text.split():
        s_score += math.log((sat_words.get(w, 0) + 1) / (sat_total + V))
        u_score += math.log((unsat_words.get(w, 0) + 1) / (unsat_total + V))

    return "Satisfied" if s_score > u_score else "Unsatisfied"

# Test data
tests = [
    "quick and helpful service",
    "poor customer support",
    "support team was polite",
    "very slow response",
    "not helpful at all",
    "happy with the service"
]

print("Naive Bayes:\n")
for t in tests:
    print(f"{t} -> {predict_nb(t)}")

# Hugging Face model
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("\nHugging Face:\n")
for t in tests:
    r = classifier(t)[0]
    label = "Satisfied" if r["label"] == "POSITIVE" else "Unsatisfied"
    print(f"{t} -> {label} ({r['score']:.2f})")

# Comparison
print("\nComparison:")
print("Feedback".ljust(30), "NB".ljust(12), "HF")

for t in tests:
    nb = predict_nb(t)
    hf = "Satisfied" if classifier(t)[0]["label"] == "POSITIVE" else "Unsatisfied"
    print(t.ljust(30), nb.ljust(12), hf)  # ✅ fixed


Naive Bayes:

quick and helpful service -> Satisfied
poor customer support -> Unsatisfied
support team was polite -> Satisfied
very slow response -> Unsatisfied
not helpful at all -> Unsatisfied
happy with the service -> Satisfied


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Hugging Face:

quick and helpful service -> Satisfied (1.00)
poor customer support -> Unsatisfied (1.00)
support team was polite -> Satisfied (0.82)
very slow response -> Unsatisfied (1.00)
not helpful at all -> Unsatisfied (1.00)
happy with the service -> Satisfied (1.00)

Comparison:
Feedback                       NB           HF
quick and helpful service      Satisfied    Satisfied
poor customer support          Unsatisfied  Unsatisfied
support team was polite        Satisfied    Satisfied
very slow response             Unsatisfied  Unsatisfied
not helpful at all             Unsatisfied  Unsatisfied
happy with the service         Satisfied    Satisfied


In [6]:
import math
import pandas as pd
from transformers import pipeline

# -----------------------------
# 1️⃣ Load Dataset
# -----------------------------
df = pd.read_csv("datasets/amazon_reviews.csv")

# Adjust column names if needed
# Assuming columns: review_text, rating
df = df[['reviewText', 'overall']].dropna()

# Convert ratings to binary sentiment
# 4,5 -> Satisfied
# 1,2 -> Unsatisfied
# Ignore rating = 3

df = df[df['overall'] != 3]

df['label'] = df['overall'].apply(
    lambda x: "Satisfied" if x >= 4 else "Unsatisfied"
)

# Split into lists
satisfied = df[df['label'] == "Satisfied"]['reviewText'].tolist()
unsatisfied = df[df['label'] == "Unsatisfied"]['reviewText'].tolist()

# -----------------------------
# 2️⃣ Word Count Function
# -----------------------------
def word_count(data):
    d = {}
    for s in data:
        for w in str(s).lower().split():
            d[w] = d.get(w, 0) + 1
    return d

sat_words = word_count(satisfied)
unsat_words = word_count(unsatisfied)

# -----------------------------
# 3️⃣ Priors
# -----------------------------
total = len(satisfied) + len(unsatisfied)
p_sat = len(satisfied) / total
p_unsat = len(unsatisfied) / total

# Vocabulary
vocab = set(sat_words) | set(unsat_words)
V = len(vocab)

sat_total = sum(sat_words.values())
unsat_total = sum(unsat_words.values())

# -----------------------------
# 4️⃣ Naive Bayes Prediction
# -----------------------------
def predict_nb(text):
    s_score = math.log(p_sat)
    u_score = math.log(p_unsat)

    for w in str(text).lower().split():
        s_score += math.log((sat_words.get(w, 0) + 1) / (sat_total + V))
        u_score += math.log((unsat_words.get(w, 0) + 1) / (unsat_total + V))

    return "Satisfied" if s_score > u_score else "Unsatisfied"

# -----------------------------
# 5️⃣ Hugging Face Model
# -----------------------------
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# -----------------------------
# 6️⃣ Test on Sample Reviews
# -----------------------------
tests = df['reviewText'].sample(5).tolist()

print("\nNaive Bayes vs HuggingFace:\n")
print("Review".ljust(100), "NB".ljust(12), "HF")

for t in tests:
    nb = predict_nb(t)
    hf_raw = classifier(t)[0]
    hf = "Satisfied" if hf_raw["label"] == "POSITIVE" else "Unsatisfied"
    
    print(t[:95].ljust(50), nb.ljust(12), hf)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Naive Bayes vs HuggingFace:

Review                                                                                               NB           HF
I use the card with GoPro Black Edition and it is just enough to record some action while horse Satisfied    Satisfied
Always bought Sandisk memories for my cameras and still satisfied. This time bought for the Gal Satisfied    Satisfied
I've just plugged and play on my Galaxy S3.It is very cheap for extra 64 Gb. (If you want the s Satisfied    Satisfied
This card works great, it has lots of space and great transfer speeds.  It shipped quickly and  Satisfied    Satisfied
Works great right out of the package, makes a great addition with the 32 GB Galaxy Note 10.1 ta Satisfied    Satisfied


In [9]:
# -----------------------------
# 7️⃣ Predict for User Review
# -----------------------------

print("\nEnter your review below:")
user_review = input("Review: ")

# Naive Bayes Prediction
nb_prediction = predict_nb(user_review)

# HuggingFace Prediction
hf_raw = classifier(user_review)[0]
hf_prediction = "Satisfied" if hf_raw["label"] == "POSITIVE" else "Unsatisfied"

print("\nPrediction Results:")
print("Naive Bayes      :", nb_prediction)
print("HuggingFace Model:", hf_prediction)
print("Confidence Score :", round(hf_raw["score"], 3))



Enter your review below:


Review:  product is bad



Prediction Results:
Naive Bayes      : Satisfied
HuggingFace Model: Unsatisfied
Confidence Score : 1.0
